# ClinVar RAG — query & summarize

Prerequisites: `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`. EDA: `clinvar_eda.ipynb`.

In [ ]:
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [ ]:
from pathlib import Path

import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
project_root = Path("..").resolve()
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

## Search

In [ ]:
query = input("Enter your ClinVar query: ").strip()
if not query:
    raise ValueError("Query cannot be empty.")

TOP_K_CHUNKS = 20
TOP_K_VARIANTS = 5

model = SentenceTransformer(EMBED_MODEL)
client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection(CLINVAR_COLLECTION)
print(f"Collection {CLINVAR_COLLECTION}: {collection.count()} chunks")

results = collection.query(
    query_embeddings=[model.encode(query).tolist()],
    n_results=TOP_K_CHUNKS,
    include=["documents", "metadatas", "distances"],
)

chunks_by_variant = {}
best_by_variant = {}
for doc_text, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0],
):
    vid = (meta or {}).get("variation_id")
    hit = {"text": doc_text, "metadata": meta, "distance": dist}
    chunks_by_variant.setdefault(vid, []).append(hit)
    if vid not in best_by_variant or dist < best_by_variant[vid]["distance"]:
        best_by_variant[vid] = hit

ranked = sorted(best_by_variant.values(), key=lambda x: x["distance"])[:TOP_K_VARIANTS]
print(f"Query: {query}\n")
for rank, hit in enumerate(ranked, 1):
    m = hit["metadata"] or {}
    n_chunks = len(chunks_by_variant.get(m.get("variation_id"), []))
    print(
        f"{rank}. variation_id={m.get('variation_id')} gene={m.get('gene')} "
        f"best_chunk={m.get('chunk_type')} dist={hit['distance']:.4f} "
        f"({n_chunks} retrieved chunk(s))"
    )
    print(hit["text"][:500] + ("..." if len(hit["text"]) > 500 else ""))
    print()

selected_ids = [int((h["metadata"] or {}).get("variation_id")) for h in ranked]
df = pd.read_parquet(CLINVAR_PARQUET)
df = df[df["VariationID"].isin(selected_ids)].copy()
print(f"Selected {len(df)} variant record(s) for context")

In [ ]:
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]


def format_variant_record(row: pd.Series) -> str:
    return "\n".join(
        f"{col}: {row[col]}"
        for col in CLINVAR_RECORD_FIELDS
        if col in row.index and pd.notna(row[col])
    )


def build_rag_context(query: str, ranked: list, chunks_by_variant: dict, records_df: pd.DataFrame) -> str:
    sections = []
    for rank, hit in enumerate(ranked, 1):
        meta = hit.get("metadata") or {}
        vid = str(meta.get("variation_id"))
        row = records_df.loc[records_df["VariationID"] == int(vid)]
        full_record = format_variant_record(row.iloc[0]) if len(row) else "(record not found)"

        variant_chunks = sorted(chunks_by_variant.get(vid, [hit]), key=lambda x: x["distance"])
        chunk_parts = []
        for i, chunk in enumerate(variant_chunks, 1):
            cm = chunk.get("metadata") or {}
            chunk_parts.append(
                f"Chunk {i} ({cm.get('chunk_type')}, dist={chunk['distance']:.4f}):\n{chunk['text']}"
            )

        sections.append(
            f"### Result {rank} — variation_id={vid}, gene={meta.get('gene')}\n\n"
            f"Retrieved chunks:\n" + "\n\n".join(chunk_parts)
            + "\n\nFull ClinVar record:\n" + full_record
        )
    return f"User query: {query}\n\n" + "\n\n---\n\n".join(sections)


rag_context = build_rag_context(query, ranked, chunks_by_variant, df)
print(rag_context[:4000] + ("..." if len(rag_context) > 4000 else ""))

## Summarize

In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LOAD_LLM = True
MAX_NEW_TOKENS = 1024

SYSTEM_PROMPT = (
    "You are a clinical genomics assistant. Summarize ClinVar variant search results "
    "for the user's query using ONLY the retrieved chunks and full records provided. "
    "Be concise, accurate, and structured. For each relevant variant, mention gene, "
    "variant name, clinical significance, phenotypes, review status, and variation ID. "
    "If information is missing, say so. Do not invent facts."
)


def summarize_rag(query: str, context: str, model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Query: {query}\n\n"
                f"Retrieved evidence:\n\n{context}\n\n"
                "Write a short summary answering the query from this evidence."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


if LOAD_LLM:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("Warning: no GPU — 7B on CPU is slow but workable (~14GB RAM).")

    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        token=hf_token,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to(device)

    summary = summarize_rag(query, rag_context, model, tokenizer)
    print("=" * 60)
    print(f"Query: {query}\n")
    print(summary)
else:
    print("LOAD_LLM is False — set True to run summarization.")